## 一、Seq2Seq的基本概念与背景

### 1. 什么是Seq2Seq？
- **定义**：Seq2Seq（Sequence-to-Sequence）是一种深度学习模型架构，专门用于处理**输入序列与输出序列长度不一致**的序列转换任务。
- **核心思想**：将输入序列先压缩为固定维度的"上下文向量"，再基于该向量逐步生成目标序列。
- **提出时间**：2014年由Sutskever等人首次提出，最初用于神经机器翻译任务。

### 2. 为什么需要Seq2Seq？
传统神经网络面临以下限制：
- **固定输入输出长度**：传统RNN假设输入长度和输出长度固定，但现实任务中：
  - 机器翻译：中文10个字可能对应英文15个词
  - 文本摘要：长文章→短摘要
  - 对话系统：长问题→短回答
- **变长映射需求**：自然语言处理核心任务（机器翻译、文本生成、摘要改写）本质都是**变长输入序列→变长输出序列**的映射问题

## 二、Seq2Seq的核心架构

#### 编码器（Encoder）
- **核心任务**：将**变长输入序列**压缩为**固定维度的上下文向量C**
- **工作原理**：
  - 逐时序吸收输入序列所有位置的语义、时序、关联信息
  - 通过RNN/LSTM/GRU逐步更新隐藏状态
  - 最终隐藏状态h作为上下文向量C
- **数学表达**：
  ```
  h_t = σ(W_xh·x_t + W_hh·h_{t-1} + b_h)
  C = h_{T_x}  (取编码器最后时刻隐状态)
  ```

#### 解码器（Decoder）
- **核心任务**：基于上下文向量C，**自回归生成目标序列**
- **工作原理**：
  - 以C作为初始状态，逐个生成目标序列token
  - 每一步生成依赖：全局上下文 + 上一时刻已生成的输出
  - 通过softmax输出词表概率分布
- **数学表达**：
  ```
  s_j = σ(W_ys·y_{j-1} + W_ss·s_{j-1} + W_cs·C + b_s)
  P(y_j|y_{1..j-1}, X) = softmax(W_vocab·s_j + b_vocab)
  ```


## 三、Seq2Seq的工作流程（以机器翻译为例）

### 1. 训练过程
- **输入**：源语言句子（如"我 爱"）
- **编码阶段**：
  - 编码器逐词读取，更新隐藏状态
  - 最终状态C = h（代表"我爱"的语义）
- **解码阶段**：
  - **教师强制（Teacher Forcing）**：使用真实目标序列作为输入
  - 解码器以C为初始状态，逐词生成目标序列（如"I love"）
  - 每一步使用真实前序词作为输入，避免误差累积

### 2. 推理过程
- **输入**：新源语言句子
- **编码阶段**：同训练过程
- **解码阶段**：
  - **自回归生成**：使用模型自身预测作为下一步输入
  - 从开始，逐步生成y, y...直到
  - 可采用贪心解码或束搜索（Beam Search）